In [14]:
import math
import pathlib
from datetime import datetime, timedelta

import matplotlib.pylab as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from scipy.stats import gaussian_kde
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    median_absolute_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from mlflow.models import infer_signature

import pandas as pd
import plotly.graph_objects as go

import mlflow

In [2]:
def generate_apple_sales_data_with_promo_adjustment(
    base_demand: int = 1000,
    n_rows: int = 5000,
    competitor_price_effect: float = -50.0,
):
    """
    Generates a synthetic dataset for predicting apple sales demand with multiple
    influencing factors.

    This function creates a pandas DataFrame with features relevant to apple sales.
    The features include date, average_temperature, rainfall, weekend flag, holiday flag,
    promotional flag, price_per_kg, competitor's price, marketing intensity, stock availability,
    and the previous day's demand. The target variable, 'demand', is generated based on a
    combination of these features with some added noise.

    Args:
        base_demand (int, optional): Base demand for apples. Defaults to 1000.
        n_rows (int, optional): Number of rows (days) of data to generate. Defaults to 5000.
        competitor_price_effect (float, optional): Effect of competitor's price being lower
                                                   on our sales. Defaults to -50.

    Returns:
        pd.DataFrame: DataFrame with features and target variable for apple sales prediction.

    Example:
        >>> df = generate_apple_sales_data_with_promo_adjustment(base_demand=1200, n_rows=6000)
        >>> df.head()
    """

    # Set seed for reproducibility
    np.random.seed(9999)

    # Create date range
    dates = [datetime.now() - timedelta(days=i) for i in range(n_rows)]
    dates.reverse()

    # Generate features
    df = pd.DataFrame(
        {
            "date": dates,
            "average_temperature": np.random.uniform(10, 35, n_rows),
            "rainfall": np.random.exponential(5, n_rows),
            "weekend": [(date.weekday() >= 5) * 1 for date in dates],
            "holiday": np.random.choice([0, 1], n_rows, p=[0.97, 0.03]),
            "price_per_kg": np.random.uniform(0.5, 3, n_rows),
            "month": [date.month for date in dates],
        }
    )

    # Introduce inflation over time (years)
    df["inflation_multiplier"] = 1 + (df["date"].dt.year - df["date"].dt.year.min()) * 0.03

    # Incorporate seasonality due to apple harvests
    df["harvest_effect"] = np.sin(2 * np.pi * (df["month"] - 3) / 12) + np.sin(
        2 * np.pi * (df["month"] - 9) / 12
    )

    # Modify the price_per_kg based on harvest effect
    df["price_per_kg"] = df["price_per_kg"] - df["harvest_effect"] * 0.5

    # Adjust promo periods to coincide with periods lagging peak harvest by 1 month
    peak_months = [4, 10]  # months following the peak availability
    df["promo"] = np.where(
        df["month"].isin(peak_months),
        1,
        np.random.choice([0, 1], n_rows, p=[0.85, 0.15]),
    )

    # Generate target variable based on features
    base_price_effect = -df["price_per_kg"] * 50
    seasonality_effect = df["harvest_effect"] * 50
    promo_effect = df["promo"] * 200

    df["demand"] = (
        base_demand
        + base_price_effect
        + seasonality_effect
        + promo_effect
        + df["weekend"] * 300
        + np.random.normal(0, 50, n_rows)
    ) * df["inflation_multiplier"]  # adding random noise

    # Add previous day's demand
    df["previous_days_demand"] = df["demand"].shift(1)
    df["previous_days_demand"].fillna(method="bfill", inplace=True)  # fill the first row

    # Introduce competitor pricing
    df["competitor_price_per_kg"] = np.random.uniform(0.5, 3, n_rows)
    df["competitor_price_effect"] = (
        df["competitor_price_per_kg"] < df["price_per_kg"]
    ) * competitor_price_effect

    # Stock availability based on past sales price (3 days lag with logarithmic decay)
    log_decay = -np.log(df["price_per_kg"].shift(3) + 1) + 2
    df["stock_available"] = np.clip(log_decay, 0.7, 1)

    # Marketing intensity based on stock availability
    # Identify where stock is above threshold
    high_stock_indices = df[df["stock_available"] > 0.95].index

    # For each high stock day, increase marketing intensity for the next week
    for idx in high_stock_indices:
        df.loc[idx : min(idx + 7, n_rows - 1), "marketing_intensity"] = np.random.uniform(0.7, 1)

    # If the marketing_intensity column already has values, this will preserve them;
    #  if not, it sets default values
    fill_values = pd.Series(np.random.uniform(0, 0.5, n_rows), index=df.index)
    df["marketing_intensity"].fillna(fill_values, inplace=True)

    # Adjust demand with new factors
    df["demand"] = df["demand"] + df["competitor_price_effect"] + df["marketing_intensity"]

    # Drop temporary columns
    df.drop(
        columns=[
            "inflation_multiplier",
            "harvest_effect",
            "month",
            "competitor_price_effect",
            "stock_available",
        ],
        inplace=True,
    )

    return df

In [3]:
my_data = generate_apple_sales_data_with_promo_adjustment(
    base_demand=1000, n_rows=10_000, competitor_price_effect=-25.0
)

/tmp/ipykernel_1584353/1129670120.py:85: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["previous_days_demand"].fillna(method="bfill", inplace=True)  # fill the first row
/tmp/ipykernel_1584353/1129670120.py:85: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["previous_days_demand"].fillna(method="bfill", inplace=True)  # fill the first row
/tmp/ipykernel_1584353/1129670120.py:108: FutureWarning: A value is trying to be set on a copy of

In [4]:
def plot_time_series_demand(data, window_size=7, plot_size=(1000, 700)):
    if not isinstance(data, pd.DataFrame):
        raise TypeError("data must be a pandas DataFrame.")

    df = data.copy()

    df["date"] = pd.to_datetime(df["date"])

    # Calculate the rolling average
    df["rolling_avg"] = df["demand"].rolling(window=window_size).mean()

    # Create the figure
    fig = go.Figure()

    # Plot the original time series data with low alpha (transparency)
    fig.add_trace(go.Scatter(
        x=df["date"],
        y=df["demand"],
        mode='lines+markers',
        name='Original Demand',
        line=dict(color='blue', width=2, dash='solid'),
        marker=dict(size=6, opacity=0.4),
        opacity=0.5
    ))

    # Plot the rolling average
    fig.add_trace(go.Scatter(
        x=df["date"],
        y=df["rolling_avg"],
        mode='lines',
        name=f'{window_size}-Day Rolling Average',
        line=dict(color='red', width=2)
    ))

    # Set layout
    fig.update_layout(
        title=f"Time Series Plot of Demand with {window_size}-Day Rolling Average",
        xaxis_title="Date",
        yaxis_title="Demand",
        template="plotly_white",
        width=plot_size[0],
        height=plot_size[1],
        legend=dict(
            title="Legend",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )

    return fig

In [5]:
def plot_box_weekend(df, plot_size=(800, 600)):
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame.")

    # Создаем фигуру
    fig = go.Figure()

    # Добавляем коробчатую диаграмму
    fig.add_trace(go.Box(
        x=df["weekend"],
        y=df["demand"],
        name="Box Plot",
        marker_color='lightgray',
        boxmean=True  # Показывает среднее значение
    ))

    # Добавляем точки (рассеивание)
    weekend_colors = {0: "blue", 1: "green"}
    for weekend_val in [0, 1]:
        filtered_df = df[df["weekend"] == weekend_val]

        # Добавляем джиттер (случайные сдвиги по оси X и Y)
        jitter_x = np.random.normal(0, 0.1, size=len(filtered_df))
        jitter_y = np.random.normal(0, 0.1, size=len(filtered_df))

        fig.add_trace(go.Scatter(
            x=filtered_df["weekend"] + jitter_x,  # Добавляем джиттер к оси X
            y=filtered_df["demand"] + jitter_y,        # Добавляем джиттер к оси Y
            mode='markers',
            name=f"Weekend {weekend_val}",
            marker=dict(
                color=weekend_colors[weekend_val],
                size=5,
                opacity=0.2,
                line=dict(width=0.5, color='darkgray')
            ),
        ))

    # Обновляем макет
    fig.update_layout(
        title="Box Plot of Demand on Weekends vs. Weekdays",
        xaxis=dict(
            title="Weekend (0: No, 1: Yes)",
            tickmode='array',
            tickvals=[0, 1],
            ticktext=["Weekday", "Weekend"]
        ),
        yaxis_title="Demand",
        template="plotly_white",
        width=plot_size[0],
        height=plot_size[1],
        showlegend=True,
        legend=dict(
            title="Legend",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        )
    )

    return fig

In [6]:
def plot_scatter_demand_price(df, plot_size=(800, 600)):
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame.")
    
    fig = go.Figure()

    # Цвета для групп (выходные и будни)
    weekend_colors = {0: "blue", 1: "green"}

    # Добавляем точки для каждой группы
    for weekend_val in [0, 1]:
        filtered_df = df[df["weekend"] == weekend_val]
        fig.add_trace(go.Scatter(
            x=filtered_df["price_per_kg"],
            y=filtered_df["demand"],
            mode='markers',
            name=f"Weekend {weekend_val}",
            marker=dict(
                color=weekend_colors[weekend_val],
                size=7,
                opacity=0.6
            )
        ))

        # Построение регрессионной линии
        if len(filtered_df) > 1:  # Проверяем, есть ли данные для построения регрессии
            X = sm.add_constant(filtered_df["price_per_kg"])  # Добавляем константу для регрессии
            model = sm.OLS(filtered_df["demand"], X).fit()
            x_range = np.linspace(filtered_df["price_per_kg"].min(), filtered_df["price_per_kg"].max(), 100)
            y_pred = model.predict(sm.add_constant(x_range))
            fig.add_trace(go.Scatter(
                x=x_range,
                y=y_pred,
                mode='lines',
                name=f"Regression Line (Weekend {weekend_val})",
                line=dict(color=weekend_colors[weekend_val], width=2)
            ))

    # Обновляем макет
    fig.update_layout(
        title="Scatter Plot of Demand vs Price per kg with Regression Line",
        xaxis_title="Price per kg",
        yaxis_title="Demand",
        template="plotly_white",
        width=plot_size[0],
        height=plot_size[1],
        legend=dict(
            title="Legend",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        )
    )

    return fig

In [7]:
def plot_density_weekday_weekend(df, plot_size=(800, 600)):
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame.")

    fig = go.Figure()

    # Цвета для групп
    weekend_colors = {0: "blue", 1: "green"}
    labels = {0: "Weekday", 1: "Weekend"}

    for weekend_val in [0, 1]:
        # Отфильтровываем данные для текущей группы
        group_data = df[df["weekend"] == weekend_val]["demand"]

        if len(group_data) > 1:  # Проверяем, есть ли достаточно данных для оценки плотности
            # Оценка плотности с использованием KDE
            kde = gaussian_kde(group_data)
            x_range = np.linspace(group_data.min(), group_data.max(), 100)
            y_values = kde(x_range)

            # Добавляем график плотности
            fig.add_trace(go.Scatter(
                x=x_range,
                y=y_values,
                mode='lines',
                name=labels[weekend_val],
                line=dict(color=weekend_colors[weekend_val], width=2),
                fill='tozeroy',
                opacity=0.6
            ))

    # Обновляем макет
    fig.update_layout(
        title="Density Plot of Demand by Weekday/Weekend",
        xaxis_title="Demand",
        yaxis_title="Density",
        template="plotly_white",
        width=plot_size[0],
        height=plot_size[1],
        legend=dict(
            title="Legend",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        )
    )

    return fig

In [8]:
def plot_coefficients(model, feature_names, plot_size=(800, 600)):
    if not hasattr(model, "coef_"):
        raise AttributeError("The model must have a 'coef_' attribute.")
    
    # Получаем коэффициенты модели
    coefficients = model.coef_

    # Создаем график
    fig = go.Figure()

    # Добавляем горизонтальный бар-чарт
    fig.add_trace(go.Bar(
        x=coefficients,
        y=feature_names,
        orientation='h',
        marker=dict(color=coefficients, colorscale="Viridis"),
        name="Coefficients"
    ))

    # Настраиваем макет
    fig.update_layout(
        title="Coefficient Plot",
        xaxis_title="Coefficient Value",
        yaxis_title="Features",
        template="plotly_white",
        width=plot_size[0],
        height=plot_size[1],
        margin=dict(l=100, r=50, t=50, b=50)
    )

    return fig

In [9]:
def plot_residuals(y_test, y_pred, plot_size=(800, 600)):
    # Вычисляем остатки
    residuals = y_test - y_pred

    # Оценка сглаженной линии для остатков
    lowess = stats.linregress(y_pred, residuals)
    line_y = lowess.intercept + lowess.slope * y_pred

    # Создаем график
    fig = go.Figure()

    # Точки для остатков
    fig.add_trace(go.Scatter(
        x=y_pred,
        y=residuals,
        mode='markers',
        name='Residuals',
        marker=dict(color="blue", opacity=0.6)
    ))

    # Добавляем линию тренда (LOWESS)
    fig.add_trace(go.Scatter(
        x=y_pred,
        y=line_y,
        mode='lines',
        name="LOWESS Line",
        line=dict(color="red", width=2)
    ))

    # Добавляем горизонтальную линию на уровне 0
    fig.add_trace(go.Scatter(
        x=[min(y_pred), max(y_pred)],
        y=[0, 0],
        mode='lines',
        name="Zero Line",
        line=dict(color="black", dash="dash")
    ))

    # Настроить макет
    fig.update_layout(
        title="Residual Plot",
        xaxis_title="Predicted values",
        yaxis_title="Residuals",
        template="plotly_white",
        width=plot_size[0],
        height=plot_size[1],
        margin=dict(l=100, r=50, t=50, b=50)
    )

    return fig

In [10]:
def plot_prediction_error(y_test, y_pred, plot_size=(800, 600)):
    # Вычисляем ошибки
    errors = y_test - y_pred

    # Создаем график
    fig = go.Figure()

    # Точки ошибок предсказания
    fig.add_trace(go.Scatter(
        x=y_pred,
        y=errors,
        mode='markers',
        name='Prediction Errors',
        marker=dict(color="blue", opacity=0.6)
    ))

    # Добавляем горизонтальную линию на уровне 0
    fig.add_trace(go.Scatter(
        x=[min(y_pred), max(y_pred)],
        y=[0, 0],
        mode='lines',
        name="Zero Error Line",
        line=dict(color="red", dash="dash")
    ))

    # Настроить макет
    fig.update_layout(
        title="Prediction Error Plot",
        xaxis_title="Predicted Values",
        yaxis_title="Errors",
        template="plotly_white",
        width=plot_size[0],
        height=plot_size[1],
        margin=dict(l=100, r=50, t=50, b=50)
    )

    return fig

In [11]:
def plot_qq(y_test, y_pred, plot_size=(800, 600)):
    # Вычисляем остатки
    residuals = y_test - y_pred

    # Создаем график
    fig = go.Figure()

    qq = stats.probplot(residuals, dist="norm")
    x = np.array([qq[0][0][0], qq[0][0][-1]])

    # Добавляем точки QQ-графика
    fig.add_trace(go.Scatter(
        x=qq[0][0],
        y=qq[0][1],
        mode='markers',
        name='QQ Plot',
        marker=dict(color="blue", opacity=0.4)
    ))

    # Добавляем линию 45 градусов (для идеального нормального распределения)
    fig.add_trace(go.Scatter(
        x=x,
        y=qq[1][1] + qq[1][0]*x,
        mode='lines',
        name="Ideal Line",
        line=dict(color="red", dash="dash")
    ))

    # Настроить макет
    fig.update_layout(
        title="QQ Plot",
        xaxis_title="Theoretical Quantiles",
        yaxis_title="Sample Quantiles",
        template="plotly_white",
        width=plot_size[0],
        height=plot_size[1],
        margin=dict(l=100, r=50, t=50, b=50)
    )

    return fig

In [12]:
def plot_correlation_matrix(df, plot_size=(800, 800)):
    # Calculate the correlation matrix
    corr = df.corr()

    # Generate a mask for the upper triangle
    mask = np.triu(np.ones_like(corr, dtype=bool))
    masked_corr = corr.mask(mask).iloc[::-1]

    text_annotations = masked_corr.round(2).fillna("").astype(str)

    # Create heatmap with annotations
    fig = go.Figure(
        data=go.Heatmap(
            z=masked_corr.values,
            x=corr.columns,
            y=corr.columns[::-1],
            colorscale="rdbu_r",
            zmin=-1,
            zmax=1,
            text=text_annotations,  # Add annotations
            texttemplate="%{text}",
            textfont=dict(size=13),
            hoverinfo="z",  # Display annotations on hover
        )
    )

    # Update layout for better aesthetics
    fig.update_layout(
        title="Feature Correlation Matrix",
        xaxis=dict(title="Features", tickangle=45, automargin=True),
        yaxis=dict(title="Features", automargin=True),
        autosize=False,
        width=plot_size[0],
        height=plot_size[1],
        coloraxis_colorbar=dict(title="Correlation", tickvals=[-1, 0, 1]),
    )

    return fig

In [18]:
import logging
logger = logging.getLogger("mlflow.models.evaluation.evaluators.shap")
logger.setLevel(logging.DEBUG)

In [24]:
mlflow.set_tracking_uri("http://127.0.0.1:5801")

mlflow.set_experiment("Visualizations Demo")

X = my_data.drop(columns=["demand", "date"])
y = my_data["demand"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

fig1 = plot_time_series_demand(my_data, window_size=28)
fig2 = plot_box_weekend(my_data)
fig3 = plot_scatter_demand_price(my_data)
fig4 = plot_density_weekday_weekend(my_data)

# Execute the correlation plot, saving the plot to a local temporary directory
fig9 = plot_correlation_matrix(my_data)

# Define our Ridge model
model = Ridge(alpha=1.0)

# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Create a model signature
signature = infer_signature(X_test, y_pred)

# Build the Evaluation Dataset from the test set
eval_data = X_test
eval_data["label"] = y_test
eval_data["predictions"] = y_pred
dataset = mlflow.data.from_pandas(
    eval_data,
    targets="label",
    predictions="predictions"
)

# Calculate error metrics
mse = mean_squared_error(y_test, y_pred)
rmse = math.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
msle = mean_squared_log_error(y_test, y_pred)
medae = median_absolute_error(y_test, y_pred)

# Generate prediction-dependent plots
fig5 = plot_residuals(y_test, y_pred)
fig6 = plot_coefficients(model, X_test.columns)
fig7 = plot_prediction_error(y_test, y_pred)
fig8 = plot_qq(y_test, y_pred)

# Start an MLflow run for logging metrics, parameters, the model, and our figures
with mlflow.start_run() as run:
    # Log the model
    mlflow.sklearn.log_model(
        sk_model=model,
        input_example=X_test,
        artifact_path="model",
        signature=signature,
    )

    # Log the metrics
    mlflow.log_metrics(
        {"mse": mse, "rmse": rmse, "mae": mae, "r2": r2, "msle": msle, "medae": medae}
    )

    # Log the hyperparameter
    mlflow.log_param("alpha", 1.0)

    # Log plots
    mlflow.log_figure(fig1, "time_series_demand.html")
    mlflow.log_figure(fig2, "box_weekend.html")
    mlflow.log_figure(fig3, "scatter_demand_price.html")
    mlflow.log_figure(fig4, "density_weekday_weekend.html")
    mlflow.log_figure(fig5, "residuals_plot.html")
    mlflow.log_figure(fig6, "coefficients_plot.html")
    mlflow.log_figure(fig7, "prediction_errors.html")
    mlflow.log_figure(fig8, "qq_plot.html")
    mlflow.log_figure(fig9, "corr_plot.html")

    # Evaluation
    mlflow.evaluate(
        data=dataset,
        targets=None,
        predictions=None,
        # model_type="regressor",
        evaluators=["default"],
    )

/home/lenow/.cache/pypoetry/virtualenvs/test-project-UdmU02IV-py3.11/lib/python3.11/site-packages/mlflow/types/utils.py:435: UserWarning:

Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.

2025/01/03 19:39:38 WARNING mlflow.models.utils: Found extra inputs in the model input that are not defined in the model signature: `['label', 'predictions']`. These inputs will be ignored.


🏃 View run lyrical-conch-416 at: http://127.0.0.1:5801/#/experiments/16/runs/eb1cadc211224cdbaca743041d309444
🧪 View experiment at: http://127.0.0.1:5801/#/experiments/16


MlflowException: The extra_metrics argument must be specified model_type is None.

In [28]:
import mlflow
import plotly.graph_objects as go

# Начинаем эксперимент в MLflow
with mlflow.start_run():
    # Создаем график с использованием Plotly
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=[1, 2, 3], y=[4, 5, 6], mode='lines+markers', name='Line 1'))
    fig.add_trace(go.Scatter(x=[1, 2, 3], y=[6, 5, 4], mode='lines+markers', name='Line 2'))
    fig.update_layout(title="Пример графика Plotly", xaxis_title="X-axis", yaxis_title="Y-axis")
    
    # Логируем график в качестве артефакта
    mlflow.log_figure(fig, "plotly_chart.html")

    # Логируем другие параметры или метрики, если нужно
    mlflow.log_param("example_param", "value")
    mlflow.log_metric("example_metric", 0.95)

print("График сохранен в MLflow.")

🏃 View run bright-seal-539 at: http://127.0.0.1:5801/#/experiments/16/runs/7546740d7c6749fb9b42c4385d560d1a
🧪 View experiment at: http://127.0.0.1:5801/#/experiments/16
График сохранен в MLflow.


In [25]:
import shap
import xgboost
from sklearn.model_selection import train_test_split

mlflow.set_experiment("TEST Shap")

# Load the UCI Adult Dataset
X, y = shap.datasets.adult()

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Fit an XGBoost binary classifier on the training data split
model = xgboost.XGBClassifier().fit(X_train, y_train)

# Build the Evaluation Dataset from the test set
y_test_pred = model.predict(X=X_test)
eval_data = X_test
eval_data["label"] = y_test
eval_data["predictions"] = y_test_pred


with mlflow.start_run() as run:
    # Evaluate the static dataset without providing a model
    result = mlflow.evaluate(
        data=eval_data,
        targets="label",
        predictions="predictions",
        model_type="classifier",
    )

print(f"metrics:\n{result.metrics}")
print(f"artifacts:\n{result.artifacts}")

2025/01/03 19:41:05 INFO mlflow.tracking.fluent: Experiment with name 'TEST Shap' does not exist. Creating a new experiment.
/home/lenow/.cache/pypoetry/virtualenvs/test-project-UdmU02IV-py3.11/lib/python3.11/site-packages/mlflow/types/utils.py:435: UserWarning:

Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.

2025/01/03 19:41:06 INFO mlflow.models.evaluation.evaluators.classifier: The eva

🏃 View run unruly-dove-238 at: http://127.0.0.1:5801/#/experiments/18/runs/f5406030a9e34b0c8d6fd044533d14f4
🧪 View experiment at: http://127.0.0.1:5801/#/experiments/18
metrics:
{'true_negatives': 7664, 'false_positives': 532, 'false_negatives': 868, 'true_positives': 1682, 'example_count': 10746, 'accuracy_score': 0.8697189651963522, 'recall_score': 0.659607843137255, 'precision_score': 0.7597109304426377, 'f1_score': 0.7061293031066331}
artifacts:
{'confusion_matrix': ImageEvaluationArtifact(uri='mlflow-artifacts:/18/f5406030a9e34b0c8d6fd044533d14f4/artifacts/confusion_matrix.png')}


<Figure size 1050x700 with 0 Axes>

In [27]:
import xgboost
import shap
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from mlflow.models import MetricThreshold

In [31]:
# load UCI Adult Data Set; segment it into training and test sets
X, y = shap.datasets.adult()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

# construct an evaluation dataset from the test set
eval_data = X_test
eval_data["label"] = y_test

In [32]:
# Log and evaluate the candidate model
candidate_model = xgboost.XGBClassifier().fit(X_train, y_train)

with mlflow.start_run(run_name="candidate") as run:
    candidate_model_uri = mlflow.sklearn.log_model(
        candidate_model, "candidate_model", signature=infer_signature(X_test, y_test)
    ).model_uri

    candidate_result = mlflow.evaluate(
        candidate_model_uri,
        eval_data,
        targets="label",
        model_type="classifier",
    )

/home/lenow/.cache/pypoetry/virtualenvs/test-project-UdmU02IV-py3.11/lib/python3.11/site-packages/mlflow/types/utils.py:435: UserWarning:

Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.

/home/lenow/.cache/pypoetry/virtualenvs/test-project-UdmU02IV-py3.11/lib/python3.11/site-packages/mlflow/types/utils.py:435: UserWarning:

Hint: Inferred schema contains integer column(s). Integer columns 

🏃 View run candidate at: http://127.0.0.1:5801/#/experiments/18/runs/1b17299a0d9444398b3276306c05e85e
🧪 View experiment at: http://127.0.0.1:5801/#/experiments/18


In [33]:

# Log and evaluate the baseline model
baseline_model = DummyClassifier(strategy="uniform").fit(X_train, y_train)

with mlflow.start_run(run_name="baseline") as run:
    baseline_model_uri = mlflow.sklearn.log_model(
        baseline_model, "baseline_model", signature=infer_signature(X_test, y_test)
    ).model_uri

    baseline_result = mlflow.evaluate(
        baseline_model_uri,
        eval_data,
        targets="label",
        model_type="classifier",
    )

/home/lenow/.cache/pypoetry/virtualenvs/test-project-UdmU02IV-py3.11/lib/python3.11/site-packages/mlflow/types/utils.py:435: UserWarning:

Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.

/home/lenow/.cache/pypoetry/virtualenvs/test-project-UdmU02IV-py3.11/lib/python3.11/site-packages/mlflow/types/utils.py:435: UserWarning:

Hint: Inferred schema contains integer column(s). Integer columns 

🏃 View run baseline at: http://127.0.0.1:5801/#/experiments/18/runs/23d5b0f85ebf4aab929b1ea2d664acf6
🧪 View experiment at: http://127.0.0.1:5801/#/experiments/18


In [39]:

# Define criteria for model to be validated against
thresholds = {
    "accuracy_score": MetricThreshold(
        threshold=0.8,  # accuracy should be >=0.8
        min_absolute_change=0.05,  # accuracy should be at least 0.05 greater than baseline model accuracy
        min_relative_change=0.05,  # accuracy should be at least 5 percent greater than baseline model accuracy
        greater_is_better=True,
    ),
}

# Validate the candidate model agaisnt baseline
result_val= mlflow.validate_evaluation_results(
    candidate_result=candidate_result,
    baseline_result=baseline_result,
    validation_thresholds=thresholds,
)

2025/01/03 22:19:27 INFO mlflow.models.evaluation.validation: Validating candidate model metrics against baseline
2025/01/03 22:19:27 INFO mlflow.models.evaluation.validation: Model validation passed!


In [41]:
mlflow.set_experiment("Evaluation GPT")

<Experiment: artifact_location='mlflow-artifacts:/19', creation_time=1735932262916, experiment_id='19', last_update_time=1735932262916, lifecycle_stage='active', name='Evaluation GPT', tags={}>

In [42]:
import xgboost
import shap
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
import mlflow
from mlflow.models.signature import infer_signature
from mlflow.models.evaluation import MetricThreshold
import pandas as pd

# Загрузка данных UCI Adult Data Set; разделение на train и test
X, y = shap.datasets.adult()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

# Подготовка данных для оценки
eval_data = X_test.copy()
eval_data["label"] = y_test
eval_data_path = "eval_data.csv"
eval_data.to_csv(eval_data_path, index=False)  # Сохраняем данные для dataset_path

# Логирование и оценка модели-кандидата
candidate_model = xgboost.XGBClassifier().fit(X_train, y_train)
candidate_signature = infer_signature(X_train, candidate_model.predict(X_train))

with mlflow.start_run(run_name="candidate") as run:
    candidate_model_uri = mlflow.sklearn.log_model(
        candidate_model, "candidate_model", signature=candidate_signature
    ).model_uri

    candidate_result = mlflow.evaluate(
        model=candidate_model_uri,
        data=eval_data,
        dataset_path=eval_data_path,  # Указываем путь к данным
        targets="label",
        model_type="classifier",
        evaluators=["default"],  # Указываем стандартный набор метрик
    )

# Логирование и оценка базовой модели
baseline_model = DummyClassifier(strategy="uniform").fit(X_train, y_train)
baseline_signature = infer_signature(X_train, baseline_model.predict(X_train))

with mlflow.start_run(run_name="baseline") as run:
    baseline_model_uri = mlflow.sklearn.log_model(
        baseline_model, "baseline_model", signature=baseline_signature
    ).model_uri

    baseline_result = mlflow.evaluate(
        model=baseline_model_uri,
        data=eval_data,
        dataset_path=eval_data_path,  # Указываем путь к данным
        targets="label",
        model_type="classifier",
        evaluators=["default"],
    )

# Определение критериев для валидации модели
thresholds = {
    "accuracy_score": MetricThreshold(
        threshold=0.8,  # accuracy >= 0.8
        min_absolute_change=0.05,  # accuracy >= baseline accuracy + 0.05
        min_relative_change=0.05,  # accuracy >= 5% больше baseline accuracy
        greater_is_better=True,
    ),
}

# Валидация модели-кандидата относительно baseline
mlflow.validate_evaluation_results(
    candidate_result=candidate_result,
    baseline_result=baseline_result,
    validation_thresholds=thresholds,
)


/home/lenow/.cache/pypoetry/virtualenvs/test-project-UdmU02IV-py3.11/lib/python3.11/site-packages/mlflow/types/utils.py:435: UserWarning:

Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.

/home/lenow/.cache/pypoetry/virtualenvs/test-project-UdmU02IV-py3.11/lib/python3.11/site-packages/mlflow/types/utils.py:435: UserWarning:

Hint: Inferred schema contains integer column(s). Integer columns 

🏃 View run candidate at: http://127.0.0.1:5801/#/experiments/19/runs/af0a1d20b8ff40d2996bf9ab19183a47
🧪 View experiment at: http://127.0.0.1:5801/#/experiments/19


/home/lenow/.cache/pypoetry/virtualenvs/test-project-UdmU02IV-py3.11/lib/python3.11/site-packages/mlflow/types/utils.py:435: UserWarning:

Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.

/home/lenow/.cache/pypoetry/virtualenvs/test-project-UdmU02IV-py3.11/lib/python3.11/site-packages/mlflow/types/utils.py:435: UserWarning:

Hint: Inferred schema contains integer column(s). Integer columns 

🏃 View run baseline at: http://127.0.0.1:5801/#/experiments/19/runs/cd002541658743ab9f589e1bcfec1f34
🧪 View experiment at: http://127.0.0.1:5801/#/experiments/19


In [43]:
import xgboost
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import mlflow

# Загрузка данных UCI Adult Data Set; разделение на train и test
X, y = shap.datasets.adult()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

# Обучение модели
model = xgboost.XGBClassifier()
model.fit(X_train, y_train)

# Предсказания
y_pred = model.predict(X_test)

# Вычисление метрик
metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1 Score": f1_score(y_test, y_pred),
}

# Создание DataFrame для логирования
eval_table = pd.DataFrame.from_dict(
    {"Metric": list(metrics.keys()), "Value": list(metrics.values())}
)

# Логирование таблицы в MLflow
with mlflow.start_run(run_name="Evaluation Table Logging"):
    mlflow.log_table(data=eval_table, artifact_file="evaluation_table.json")


🏃 View run Evaluation Table Logging at: http://127.0.0.1:5801/#/experiments/19/runs/6bd4ae02f5d64822aa4d88edacf15f95
🧪 View experiment at: http://127.0.0.1:5801/#/experiments/19
